# Overlapping Users

The notebook will contain information on how many users are overlapping between consecutive time windows for different aggregation levels.

In [1]:
# Main packages 
import polars as pl
import matplotlib.pyplot as plt
import numpy as np

# Clustering packages
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score, calinski_harabasz_score, davies_bouldin_score,adjusted_rand_score

# Parallel processing packages
from joblib import Parallel, delayed

In [2]:
# Load data
df = pl.scan_csv("/home/lanl/data/cyber1/auth.txt.gz",has_header=False,separator=",",
                 new_columns= ['time','src_user','dest_user','src_comp','dest_comp',
                               'auth_type','logon_type','auth_orientation','outcome'])

In [3]:
# Keep only human users
df = df.filter(pl.col("src_user").str.starts_with("U"))

## Functions & Global info

In [4]:
# Time conversions
seconds_in_day = 60 * 60 * 24

# Time aggregation
agg_hour_level = 1

# Number of buckets in a week of data
buckets_per_week = (7 * 24) // agg_hour_level


In [5]:
# FUNCTION - build features dataframe
def build_features(df,agg_hour_level):

    agg_seconds = agg_hour_level * 60 * 60

    return (
        df.with_columns(
            bucket = pl.col('time') // agg_seconds,
            theta = ((pl.col('time') % seconds_in_day)/ seconds_in_day) * 2 * np.pi,
            is_failure = (pl.col('outcome') == 'Fail').cast(pl.Int8),
        )
        .group_by(['src_user','bucket'])
        .agg(
            n_events = pl.len(),
            failure_ratio = pl.col('is_failure').mean(),
            n_distinct_dest = pl.col('dest_comp').n_unique(),
            n_distinct_src = pl.col('src_comp').n_unique(),
            c_bar = pl.col('theta').cos().mean(),
            s_bar = pl.col('theta').sin().mean(),
        )
        .with_columns(
             log_n_events=pl.col("n_events").log1p(),
             log_n_distinct_dest=pl.col("n_distinct_dest").log1p(),
             log_n_distinct_src=pl.col("n_distinct_src").log1p(),
        )
        .collect()
    )

In [6]:
# Relevant feauture columns
feature_cols = [
    "log_n_events",
    "failure_ratio",
    "log_n_distinct_dest",
    "log_n_distinct_src",
    "c_bar",
    "s_bar",
]

In [7]:
# FUNCTION - process features for clustering 
def cluster_preprocess(features_df,feature_cols,week):

    lb = (week - 1) * buckets_per_week
    ub = lb + buckets_per_week - 1

    features_week = features_df.filter(pl.col('bucket').is_between(lb,ub))

    X = features_week.select(feature_cols).to_numpy()
    scaler = StandardScaler()
    X_scaled = scaler.fit_transform(X)

    return features_week, X_scaled

# 1 Hour

In [8]:
# Create features dataframe
features_df = build_features(df,agg_hour_level)

In [9]:
n_weeks = 8
weekly_results = {}

for week in range(1, n_weeks + 1):  
    features_week, _ = cluster_preprocess(features_df, feature_cols, week=week)
    
    weekly_results[week] = features_week

In [10]:
for week in range(1, n_weeks):

    w_curr = weekly_results[week].with_columns(
        relative_bucket = pl.col('bucket') % buckets_per_week
    )

    w_next = weekly_results[week + 1].with_columns(
        relative_bucket = pl.col('bucket') % buckets_per_week
    )

    overlap = w_curr.join(w_next,on = ['src_user','relative_bucket'],how = 'inner',suffix = '_next')
    
    union = len(w_curr) + len(w_next) - len(overlap)
    
    print(f"  Overlap: {len(overlap)} week {week} pairs ({len(overlap)/union:.2%})")

  Overlap: 615624 week 1 pairs (56.89%)
  Overlap: 626524 week 2 pairs (58.38%)
  Overlap: 633677 week 3 pairs (59.85%)
  Overlap: 710405 week 4 pairs (64.69%)
  Overlap: 709483 week 5 pairs (62.60%)
  Overlap: 683873 week 6 pairs (58.10%)
  Overlap: 665237 week 7 pairs (56.48%)


# 3 Hour

In [11]:
# Time aggregation
agg_hour_level = 3

# Number of buckets in a week of data
buckets_per_week = (7 * 24) // agg_hour_level

In [12]:
# Create features dataframe
features_df = build_features(df,agg_hour_level)

In [13]:
n_weeks = 8
weekly_results = {}

for week in range(1, n_weeks + 1):  
    features_week, _ = cluster_preprocess(features_df, feature_cols, week=week)
    
    weekly_results[week] = features_week

In [14]:
for week in range(1, n_weeks):

    w_curr = weekly_results[week].with_columns(
        relative_bucket = pl.col('bucket') % buckets_per_week
    )

    w_next = weekly_results[week + 1].with_columns(
        relative_bucket = pl.col('bucket') % buckets_per_week
    )

    overlap = w_curr.join(w_next,on = ['src_user','relative_bucket'],how = 'inner',suffix = '_next')
    
    union = len(w_curr) + len(w_next) - len(overlap)
    
    print(f"  Overlap: {len(overlap)} week {week} pairs ({len(overlap)/union:.2%})")

  Overlap: 260254 week 1 pairs (60.89%)
  Overlap: 265449 week 2 pairs (62.82%)
  Overlap: 268752 week 3 pairs (64.51%)
  Overlap: 297580 week 4 pairs (69.01%)
  Overlap: 291873 week 5 pairs (66.13%)
  Overlap: 272054 week 6 pairs (59.89%)
  Overlap: 265632 week 7 pairs (58.26%)


# 6 Hour

In [15]:
# Time aggregation
agg_hour_level = 6

# Number of buckets in a week of data
buckets_per_week = (7 * 24) // agg_hour_level

In [16]:
# Create features dataframe
features_df = build_features(df,agg_hour_level)

In [17]:
n_weeks = 8
weekly_results = {}

for week in range(1, n_weeks + 1):  
    features_week, _ = cluster_preprocess(features_df, feature_cols, week=week)
    
    weekly_results[week] = features_week

In [18]:
for week in range(1, n_weeks):

    w_curr = weekly_results[week].with_columns(
        relative_bucket = pl.col('bucket') % buckets_per_week
    )

    w_next = weekly_results[week + 1].with_columns(
        relative_bucket = pl.col('bucket') % buckets_per_week
    )

    overlap = w_curr.join(w_next,on = ['src_user','relative_bucket'],how = 'inner',suffix = '_next')
    
    union = len(w_curr) + len(w_next) - len(overlap)
    
    print(f"  Overlap: {len(overlap)} week {week} pairs ({len(overlap)/union:.2%})")

  Overlap: 141712 week 1 pairs (59.66%)
  Overlap: 143750 week 2 pairs (61.34%)
  Overlap: 145464 week 3 pairs (62.85%)
  Overlap: 160811 week 4 pairs (67.16%)
  Overlap: 158552 week 5 pairs (64.74%)
  Overlap: 147642 week 6 pairs (58.69%)
  Overlap: 145471 week 7 pairs (57.39%)
